In [1]:
import sqlite3
import pandas as pd

In [2]:
# Load datasets
athletes = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-07-27/athlete_events.csv")
regions = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-07-27/noc_regions.csv")
sales = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv")
sales = pd.read_csv("https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv", sep='\t')

In [3]:
# Connect to SQLite in-memory DB
conn = sqlite3.connect(":memory:")

In [4]:
# Write DataFrames to SQL tables
athletes.to_sql("athletes_table", conn, index=False, if_exists="replace")
regions.to_sql("regions_table", conn, index=False, if_exists="replace")
sales.to_sql("sales_table", conn, index=False, if_exists="replace")

4622

In [5]:
pd.read_sql("SELECT * FROM athletes_table LIMIT 5;", conn)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


Count the total number of medals won by each country and show the top 5.

In [6]:
pd.read_sql(
    """
    SELECT DISTINCT City,
    COUNT(Medal) AS Medal_Count
FROM athletes_table
GROUP BY City
ORDER BY Medal_Count DESC LIMIT 5
    """, conn
)

,City,Medal_Count
0,London,3624
1,Athina,2602
2,Los Angeles,2123
3,Beijing,2048
4,Rio de Janeiro,2023


Calculate the average age of athletes who won a Gold medal.


In [7]:
pd.read_sql(
    """
        SELECT AVG(Age) AS Average_Age
        FROM athletes_table
        WHERE Medal IS "Gold";
    """, conn
)

,Average_Age
0,25.901013


How many distinct events are there in each sport?

In [8]:
pd.read_sql(
    """
        SELECT DISTINCT Event, 
        Sport
        FROM athletes_table
    """, conn
)

,Event,Sport
0,Basketball Men's Basketball,Basketball
1,Judo Men's Extra-Lightweight,Judo
2,Football Men's Football,Football
3,Tug-Of-War Men's Tug-Of-War,Tug-Of-War
4,Speed Skating Women's 500 metres,Speed Skating
...,...,...
760,Weightlifting Men's All-Around Dumbbell Contest,Weightlifting
761,"Archery Men's Au Chapelet, 33 metres",Archery
762,"Archery Men's Au Cordon Dore, 33 metres",Archery
763,"Archery Men's Target Archery, 28 metres, Indiv...",Archery


Show all athletes from the United States(NOC = USA)

In [9]:
pd.read_sql(
    """
        SELECT *
        FROM athletes_table
        WHERE City = "USA"
    
""" , conn
    )

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal


Count how many medals were awarded each year.

In [10]:
pd.read_sql(
    """
            SELECT Year,
            COUNT(Medal) AS Medal_Count 
            FROM athletes_table
            WHERE Medal IS NOT NULL
            GROUP BY Year
            ORDER BY Medal_Count DESC
            """, conn  
)         

,Year,Medal_Count
0,2008,2048
1,1992,2030
2,2016,2023
3,2000,2004
4,2004,2001
5,2012,1941
6,1988,1845
7,1996,1842
8,1984,1698
9,1980,1602


Find all athletes records where height and weight is missing.

In [11]:
pd.read_sql(
    """
            SELECT *
            FROM athletes_table
            WHERE Height IS NULL AND Weight IS NULL
    """ , conn
    )

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,3,Gunnar Nielsen Aaby,M,24.0,None,None,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
1,4,Edgar Lindenau Aabye,M,34.0,None,None,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
2,10,"Einar Ferdinand ""Einari"" Aalto",M,26.0,None,None,Finland,FIN,1952 Summer,1952,Summer,Helsinki,Swimming,Swimming Men's 400 metres Freestyle,NaN
3,15,Arvo Ossian Aaltonen,M,22.0,None,None,Finland,FIN,1912 Summer,1912,Summer,Stockholm,Swimming,Swimming Men's 200 metres Breaststroke,NaN
4,15,Arvo Ossian Aaltonen,M,22.0,None,None,Finland,FIN,1912 Summer,1912,Summer,Stockholm,Swimming,Swimming Men's 400 metres Breaststroke,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58778,135539,Marius Edmund Zwiller,M,18.0,None,None,France,FRA,1924 Summer,1924,Summer,Paris,Swimming,Swimming Men's 200 metres Breaststroke,NaN
58779,135542,Werner Zwingli,M,29.0,None,None,Switzerland,SUI,1956 Winter,1956,Winter,Cortina d'Ampezzo,Cross Country Skiing,Cross Country Skiing Men's 15 kilometres,NaN
58780,135542,Werner Zwingli,M,29.0,None,None,Switzerland,SUI,1956 Winter,1956,Winter,Cortina d'Ampezzo,Cross Country Skiing,Cross Country Skiing Men's 4 x 10 kilometres R...,NaN
58781,135552,Jan (Johann-) Zybert (Siebert-),M,20.0,None,None,Poland,POL,1928 Summer,1928,Summer,Amsterdam,Cycling,"Cycling Men's Team Pursuit, 4,000 metres",NaN


Replace the missing height with the average athlete height

In [29]:
# 1. Write the correct SQL statement using a subquery
sql_update = """
UPDATE atheletes_table
SET Height = (SELECT AVG(Height) FROM atheletes_table)
WHERE Height IS NULL;
"""

In [14]:
sql_update = """
UPDATE sales_table

"""

In [15]:
pd.read_sql(
    """
        SELECT *
        FROM sales_table
    """, conn
    )


,order_id,quantity,item_name,choice_description,item_price
0,1,1,Chips and Fresh Tomato Salsa,NaN,$2.39
1,1,1,Izze,[Clementine],$3.39
2,1,1,Nantucket Nectar,[Apple],$3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,NaN,$2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98
...,...,...,...,...,...
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",$11.75
4618,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",$11.75
4619,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",$11.25
4620,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",$8.75


Return total sales value per item

In [28]:
pd.read_sql(
    """
        SELECT item_name, quantity, item_price, (quantity * item_price) AS Total_Price
        FROM sales_table
        GROUP BY item_name
    """, conn
    )
        

,item_name,quantity,item_price,Total_Price
0,6 Pack Soft Drink,1,$6.49,0
1,Barbacoa Bowl,1,$11.75,0
2,Barbacoa Burrito,1,$8.99,0
3,Barbacoa Crispy Tacos,1,$11.75,0
4,Barbacoa Salad Bowl,1,$11.89,0
5,Barbacoa Soft Tacos,1,$9.25,0
6,Bottled Water,1,$1.09,0
7,Bowl,3,$22.20,0
8,Burrito,1,$7.40,0
9,Canned Soda,2,$2.18,0


Show the top 5 records with the highest item_price

In [17]:
pd.read_sql(
    """
        SELECT *
        FROM sales_table
        ORDER BY item_price DESC
        LIMIT 5
    """, conn
    )

,order_id,quantity,item_name,choice_description,item_price
0,250,1,Steak Salad Bowl,"[Fresh Tomato Salsa, Lettuce]",$9.39
1,576,1,Barbacoa Salad Bowl,[Roasted Chili Corn Salsa],$9.39
2,576,1,Barbacoa Salad Bowl,[Roasted Chili Corn Salsa],$9.39
3,738,1,Barbacoa Salad Bowl,"[Fresh Tomato Salsa, [Rice, Pinto Beans, Chees...",$9.39
4,756,1,Carnitas Salad Bowl,"[Fresh Tomato Salsa, [Rice, Cheese, Sour Cream]]",$9.39


How many unique customer orders are there?

In [18]:
pd.read_sql(
    """
        SELECT DISTINCT item_name
        FROM sales_table
    """, conn
    )

,item_name
0,Chips and Fresh Tomato Salsa
1,Izze
2,Nantucket Nectar
3,Chips and Tomatillo-Green Chili Salsa
4,Chicken Bowl
5,Side of Chips
6,Steak Burrito
7,Steak Soft Tacos
8,Chips and Guacamole
9,Chicken Crispy Tacos


SECTION 2:
Count the number of atheletes who won at least one medal 

In [19]:
pd.read_sql(
    """
        SELECT COUNT(Medal) AS Athlete_Medal 
        FROM athletes_table
        WHERE Medal IS NOT NULL
    """, conn
    )

,Athlete_Medal
0,39783


In [20]:
pd.read_sql(
    """
   SELECT *
   FROM athletes_table
    """ , conn
)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271111,135569,Andrzej ya,M,29.0,179.0,89.0,Poland-1,POL,1976 Winter,1976,Winter,Innsbruck,Luge,Luge Mixed (Men)'s Doubles,NaN
271112,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Individual",NaN
271113,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Team",NaN
271114,135571,Tomasz Ireneusz ya,M,30.0,185.0,96.0,Poland,POL,1998 Winter,1998,Winter,Nagano,Bobsleigh,Bobsleigh Men's Four,NaN


Determine the average age of those medalists

In [21]:
pd.read_sql(
    """
        SELECT AVG(Age) AS Average_Age
        FROM athletes_table
        WHERE Medal IS NOT NULL
    """, conn
)

,Average_Age
0,25.925175


Create a new column called performance "High if average age is below 25, Medium if between 25 and 30, Low if above 30"

In [22]:
pd.read_sql(
    """
        SELECT *,
        CASE
            WHEN Age < 25 THEN 'High'
            WHEN Age BETWEEN 25 AND 30 THEN 'Medium'
            ELSE 'Low'
        END AS Performance
        FROM athletes_table
        
    """, conn
    )

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,Performance
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN,High
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN,High
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN,High
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold,Low
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271111,135569,Andrzej ya,M,29.0,179.0,89.0,Poland-1,POL,1976 Winter,1976,Winter,Innsbruck,Luge,Luge Mixed (Men)'s Doubles,NaN,Medium
271112,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Individual",NaN,Medium
271113,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Team",NaN,Medium
271114,135571,Tomasz Ireneusz ya,M,30.0,185.0,96.0,Poland,POL,1998 Winter,1998,Winter,Nagano,Bobsleigh,Bobsleigh Men's Four,NaN,Medium
